# G1 mode switching

This notebook builds a small control panel for switching the Unitree G1 between the main locomotion modes used during the academy. Run the cells from top to bottom while the robot is powered, connected to the same DDS network interface, and in a safe open area.

Mode switching sends real commands to the robot. Keep one person responsible for the emergency stop and only press buttons when the robot state shown by the panel matches what you expect.


The first code cell imports the Unitree SDK classes used by this notebook. `LocoClient` sends locomotion finite-state-machine commands, while `MotionSwitcherClient` checks or releases the higher-level motion service that can block direct locomotion control.


In [ ]:
try:
    from unitree_sdk2py.core.channel import ChannelFactoryInitialize
    from unitree_sdk2py.comm.motion_switcher.motion_switcher_client import MotionSwitcherClient
    from unitree_sdk2py.g1.loco.g1_loco_api import (
        ROBOT_API_ID_LOCO_GET_FSM_ID,
        ROBOT_API_ID_LOCO_GET_FSM_MODE,
    )
    from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
except ImportError as exc:
    raise SystemExit(
        "unitree_sdk2py is not installed. Install it with:\n"
        "  pip install -e <path-to-unitree_sdk2_python>"
    ) from exc


This cell imports standard Python helpers and notebook UI libraries. The lock protects SDK calls when several UI callbacks happen close together, and `ipywidgets` provides the buttons, status text, and details panel inside Jupyter.


In [ ]:
import json
import os
import threading
import time
from dataclasses import dataclass
from typing import Any

import ipywidgets as widgets
from IPython.display import display


These constants give readable names to the finite-state-machine IDs used by the G1 locomotion service. The error hints translate common motion-switcher return codes into messages that are easier to interpret during an exercise.


In [ ]:
FSM_ZERO_TORQUE = 0
FSM_DAMPING = 1
FSM_SIT = 3
FSM_PREPARE = 4
FSM_WALK = 501
FSM_RUN = 802
DEFAULT_CLIMB_FSM = int(os.environ.get("G1_CLIMB_FSM_ID", "812"))
AI_MODE_NAME = "ai_sport"
MODE_ALIASES = {
    "ai": AI_MODE_NAME,
}
ERROR_HINTS = {
    0: "success",
    7001: "request parameter error",
    7002: "service busy; retry",
    7004: "unsupported mode name",
    7005: "internal command execute error",
    7006: "check command execute error",
    7007: "switch command execute error",
    7008: "release command execute error",
    7009: "custom config set error",
}


`RobotState` is a small structured snapshot of what the robot reports. Keeping the state in one object makes it clear which values came from locomotion RPCs, which came from the motion switcher, and whether any part of the probe failed.


In [ ]:
@dataclass(frozen=True)
class RobotState:
    mode: str
    fsm_id: int
    fsm_mode: int
    motion_mode: str
    motion_raw: Any
    motion_code: int
    loco_skipped: bool = False
    error: str = "None"


def error_hint(code: int) -> str:
    # TODO: Turn numeric SDK return codes into short human-readable guidance.
    raise NotImplementedError("Participant exercise: complete error_hint.")


def state_text(state: RobotState) -> str:
    # TODO: Summarize the current robot mode and SDK status for display.
    raise NotImplementedError("Participant exercise: complete state_text.")


def state_detail(state: RobotState) -> str:
    # TODO: Return a structured detail dictionary for debugging mode transitions.
    raise NotImplementedError("Participant exercise: complete state_detail.")


The `Robot` class hides the raw SDK calls behind academy-friendly methods. It initializes the DDS channel lazily, reads the current mode, classifies FSM IDs into names, and exposes one method per button in the UI.


In [ ]:
class Robot:
    def __init__(self, iface="eth0", domain_id=0, timeout=10.0, climb_fsm_id=DEFAULT_CLIMB_FSM):
        # TODO: Initialize instance fields, clients, publishers/subscribers, locks, and default state needed by the class.
        raise NotImplementedError("Participant exercise: complete __init__.")

    def _ensure_clients(self):
        # TODO: Create SDK clients lazily and set their timeout/API version options.
        raise NotImplementedError("Participant exercise: complete _ensure_clients.")

    @staticmethod
    def _result_code(result):
        # TODO: Normalize SDK return objects into an integer result code.
        raise NotImplementedError("Participant exercise: complete _result_code.")

    @staticmethod
    def _rpc_get_int(client, api_id):
        # TODO: Call an RPC getter and coerce its result into an int.
        raise NotImplementedError("Participant exercise: complete _rpc_get_int.")

    @staticmethod
    def _motion_mode_name(data):
        # TODO: Read and normalize the sport/motion mode name.
        raise NotImplementedError("Participant exercise: complete _motion_mode_name.")

    @staticmethod
    def _canonical_motion_name(name):
        # TODO: Map SDK-specific motion names into notebook mode names.
        raise NotImplementedError("Participant exercise: complete _canonical_motion_name.")

    @classmethod
    def _motion_is_ai(cls, name):
        # TODO: Return whether the motion mode belongs to the AI/dev-mode family.
        raise NotImplementedError("Participant exercise: complete _motion_is_ai.")

    def _classify_mode(self, fsm_id):
        # TODO: Combine motion and low-level status into a RobotState value.
        raise NotImplementedError("Participant exercise: complete _classify_mode.")

    def get_mode(self):
        # TODO: Query SDK state, classify it, and include raw fields for debugging.
        raise NotImplementedError("Participant exercise: complete get_mode.")

    def switch_mode_damping(self):
        # TODO: Send the SDK command sequence for damping mode.
        raise NotImplementedError("Participant exercise: complete switch_mode_damping.")

    def switch_mode_zero_torque(self):
        # TODO: Send the SDK command sequence for zero-torque mode.
        raise NotImplementedError("Participant exercise: complete switch_mode_zero_torque.")

    def switch_mode_prepare(self):
        # TODO: Send the SDK command sequence for prepare/stand mode.
        raise NotImplementedError("Participant exercise: complete switch_mode_prepare.")

    def switch_mode_sit(self):
        # TODO: Send the SDK command sequence for sitting.
        raise NotImplementedError("Participant exercise: complete switch_mode_sit.")

    def switch_mode_walk(self):
        # TODO: Send the SDK command sequence for walking.
        raise NotImplementedError("Participant exercise: complete switch_mode_walk.")

    def switch_mode_run(self):
        # TODO: Send the SDK command sequence for running.
        raise NotImplementedError("Participant exercise: complete switch_mode_run.")

    def switch_mode_climb(self):
        # TODO: Send the SDK command sequence for stair climbing.
        raise NotImplementedError("Participant exercise: complete switch_mode_climb.")

    def switch_mode_dev(self):
        # TODO: Send the SDK command sequence for AI/dev mode.
        raise NotImplementedError("Participant exercise: complete switch_mode_dev.")

    def command(self, name):
        # TODO: Dispatch a UI command name to the corresponding switch method.
        raise NotImplementedError("Participant exercise: complete command.")


The next helpers encode the same button availability rules used by the standalone mode-control script. They are intentionally conservative: for example, walk/run/climb are enabled only when the robot is already in a locomotion-ready family of modes.


In [ ]:
def button_disabled(mode, button):
    # TODO: Decide whether each mode button should be disabled from the current robot state.
    raise NotImplementedError("Participant exercise: complete button_disabled.")


Create the robot connection object here. Change `IFACE` if your robot network is not on `eth0`, and change `DOMAIN_ID` only if your academy setup uses a non-default DDS domain.


In [ ]:
IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
TIMEOUT = 10.0

robot = Robot(iface=IFACE, domain_id=DOMAIN_ID, timeout=TIMEOUT)
print(f"Robot client configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Run this cell whenever you want a quick text-only mode check. It is useful before opening the full control panel, because it verifies that DDS discovery and the SDK clients can talk to the robot.


In [ ]:
current_state = robot.get_mode()
print(state_text(current_state))
print(state_detail(current_state))


This final cell creates the notebook control panel. The buttons call the `Robot` methods above, refresh the state display after every command, and keep a short event log so participants can see what was sent during the exercise.


In [ ]:
button_specs = [
    ("damping", "Damping", "warning"),
    ("zero_torque", "Zero Torque", "danger"),
    ("prepare", "Prepare", "primary"),
    ("sit", "Sit", "secondary"),
    ("walk", "Walk", "success"),
    ("run", "Run", "success"),
    ("climb", "Climb", "success"),
    ("dev", "Dev Off", "secondary"),
]

status_label = widgets.HTML(value="")
command_status = widgets.HTML(value="")
state_box = widgets.Textarea(
    value="",
    layout=widgets.Layout(width="100%", height="220px"),
    disabled=True,
)
event_log_box = widgets.Textarea(
    value="",
    layout=widgets.Layout(width="100%", height="120px"),
    disabled=True,
)
refresh_button = widgets.Button(description="Refresh", button_style="info")
mode_buttons = {}
event_log = []


def _set_button_style(button, style_name):
    # ipywidgets supports a limited set of Bootstrap-style names.
    # TODO: Complete the implementation for _set_button_style using the surrounding notebook context.
    raise NotImplementedError("Participant exercise: complete _set_button_style.")


for name, label, style_name in button_specs:
    button = widgets.Button(
        description=label,
        layout=widgets.Layout(width="150px", height="44px"),
    )
    _set_button_style(button, style_name)
    mode_buttons[name] = button


def refresh_panel(status=None):
    # TODO: Fetch robot state, update text/detail widgets, and refresh button styles.
    raise NotImplementedError("Participant exercise: complete refresh_panel.")


def on_command(name):
    # TODO: Run the selected robot mode command and refresh the panel with success or error details.
    raise NotImplementedError("Participant exercise: complete on_command.")


for name, button in mode_buttons.items():
    button.on_click(on_command(name))

refresh_button.on_click(lambda _button: refresh_panel("State refreshed."))

button_rows = [
    widgets.HBox([mode_buttons["damping"], mode_buttons["zero_torque"], mode_buttons["prepare"], mode_buttons["sit"]]),
    widgets.HBox([mode_buttons["walk"], mode_buttons["run"], mode_buttons["climb"], mode_buttons["dev"], refresh_button]),
]

refresh_panel("Panel ready.")
display(widgets.VBox([status_label, *button_rows, command_status, state_box, event_log_box]))
